In [1]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import numpy as np
import pandas as pd
import pickle
import ast
from functools import reduce
from tqdm.auto import tqdm
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import PandasTools
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default_config = {
    "toImageButtonOptions": {"format": "png", "scale": 5}}

# sys.path.append(os.path.abspath(R"C:\\Users\\Idener\\Sync\\CODE"))
# from aiddtools import fingerprint_tools as fpt
clear_output()

In [ ]:
df_fragments = pd.read_pickle('NPA_fragments_noann.pkl')

df_fragmented = df_fragments[df_fragments['Frag_IDs'].apply(lambda x: len(x) > 1)]
df_whole = df_fragments[df_fragments['Frag_IDs'].apply(lambda x: len(x) == 1)]

To calculate MAP4 similarity matrix and clustering:
$ python map4_clustering.py

Good, those are exactly the right levers. You just need to frame them cleanly.

What you are showing (in one line)

SOG is better because it corrects specific MAP4 errors, not because it matches MAP4 better.

Case A: disagreement analysis (core evidence)

Focus on low-purity MAP4 clusters.

For each:

MAP4 cluster = mixed under SOG

inspect SOG subclusters

What you want to show:

MAP4 groups molecules that are topologically different

SOG splits them into structurally coherent groups

If this happens consistently, your claim is:

MAP4 over-merges, SOG resolves structure.

Case C: representative examples (how you convince people)

Pick a few clean cases:

one MAP4 cluster

show 2–3 SOG subclusters

describe the structural difference:

different fragment connectivity

different topology despite shared fragments

Keep it simple and visual in a paper.

This is often more persuasive than global metrics.

Minimal supporting metric (optional but useful)

Add one number to support the story:

within-cluster node-overlap variance

or average pairwise overlap

If SOG clusters are tighter:

SOG produces more structurally coherent clusters

Final positioning

Do not argue:

“SOG matches MAP4 well”

Argue:

SOG agrees with MAP4 in simple cases but systematically refines clusters where MAP4 loses structural resolution

That is a strong, defensible claim.

## Calculate MAP4 threshold

In [ ]:
with open("NPA_map4_similarity.pkl", "rb") as f:
    data = pickle.load(f)
ids = np.array(data["ids"])
sim = np.asarray(data["sim_matrix"], dtype=np.float32)

df_NPA = pd.read_csv('/home/mattia/geass/data/NPAtlas_download_2024_09.tsv', sep='\t')

In [6]:
import re
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score

df_NPA = pd.read_csv("/home/mattia/geass/data/NPAtlas_download_2024_09.tsv", sep="\t")
df_clusters_map4 = pd.read_csv("/home/mattia/geass/NPA_analysis/NPA_clusters_map4.csv")

df_eval = df_clusters_map4.merge(
    df_NPA[["npaid", "compound_cluster_id"]].rename(columns={'npaid':'NPA_ID'}),
    on="NPA_ID", how="inner").dropna(subset=["compound_cluster_id"]).copy()

ref = df_eval["compound_cluster_id"].astype("category").cat.codes.to_numpy()

cluster_cols = [c for c in df_clusters_map4.columns if c.startswith("cluster_id_t")]

results = []

for col in cluster_cols:
    pred = df_eval[col].astype("category").cat.codes.to_numpy()

    results.append({
        "col": col,
        "t": float(col.replace("cluster_id_t", "")),
        "n_clusters": df_eval[col].nunique(),
        "ARI": adjusted_rand_score(ref, pred),
        "NMI": normalized_mutual_info_score(ref, pred),
        "V": v_measure_score(ref, pred)
    })

results = pd.DataFrame(results).sort_values("ARI", ascending=False)

print(results)

best = results.iloc[0]
print("\nbest t:", best["t"], "col:", best["col"])

               col     t  n_clusters       ARI       NMI         V
13  cluster_id_t85  85.0        1522  0.411383  0.910210  0.910210
14  cluster_id_t90  90.0         884  0.328778  0.858715  0.858715
12  cluster_id_t80  80.0        2042  0.273621  0.920880  0.920880
11  cluster_id_t75  75.0        2499  0.227505  0.921901  0.921901
10  cluster_id_t70  70.0        2891  0.190949  0.919675  0.919675
9   cluster_id_t65  65.0        3272  0.159353  0.913264  0.913264
8   cluster_id_t60  60.0        3638  0.131895  0.906015  0.906015
7   cluster_id_t55  55.0        4022  0.108789  0.898345  0.898345
6   cluster_id_t50  50.0        4424  0.076981  0.890364  0.890364
5   cluster_id_t45  45.0        4881  0.060475  0.882322  0.882322
4   cluster_id_t40  40.0        5391  0.045511  0.874254  0.874254
3   cluster_id_t35  35.0        6096  0.035103  0.865564  0.865564
2   cluster_id_t30  30.0        6972  0.026709  0.856891  0.856891
1   cluster_id_t25  25.0        9060  0.018422  0.843461  0.84

## Silhouette analysis

In [2]:
from sklearn.metrics import silhouette_score
from scipy.sparse import load_npz

df_clusters_map4 = pd.read_csv("/home/mattia/geass/NPA_analysis/NPA_clusters_map4.csv")

base = "/home/mattia/geass/NPA_analysis/NPA_sim_matrix_map4"
ids = np.load(f"{base}_ids.npy", allow_pickle=True)
sim_matrix = load_npz(f"{base}.npz").toarray().astype(np.float32, copy=False)

# Align matrix rows/cols to the cluster table order.
order = pd.Index(ids).get_indexer(df_clusters_map4["NPA_ID"].to_numpy())
if (order < 0).any():
    missing = df_clusters_map4.loc[order < 0, "NPA_ID"].head().tolist()
    raise ValueError(f"Some NPA_IDs are missing from similarity ids, e.g. {missing}")
sim_matrix = sim_matrix[np.ix_(order, order)]

dist = 1.0 - sim_matrix
dist = ((dist + dist.T) * 0.5).astype(np.float32, copy=False)
np.fill_diagonal(dist, 0.0)

cluster_cols = [c for c in df_clusters_map4.columns if c.startswith('cluster_id_t')]

results = []
for col in cluster_cols:
    labels = df_clusters_map4[col].values
    counts = pd.Series(labels).value_counts(dropna=True)
    n_clusters = int(counts.shape[0])
    avg_cluster_size = float(counts.mean()) if n_clusters > 0 else np.nan

    if n_clusters > 1:
        score = silhouette_score(dist, labels, metric='precomputed')
    else:
        score = np.nan

    results.append({
        'column': col,
        'threshold': int(col.split('t')[-1]) / 100,
        'n_clusters': n_clusters,
        'avg_cluster_size': avg_cluster_size,
        'silhouette': score
    })

df_silhouette = pd.DataFrame(results).sort_values('threshold')
print(df_silhouette)

            column  threshold  n_clusters  avg_cluster_size  silhouette
0   cluster_id_t20       0.20       10952          1.536158    0.445773
1   cluster_id_t25       0.25        9060          1.856954    0.511030
2   cluster_id_t30       0.30        6972          2.413081    0.580667
3   cluster_id_t35       0.35        6096          2.759843    0.596493
4   cluster_id_t40       0.40        5391          3.120757    0.598626
5   cluster_id_t45       0.45        4881          3.446835    0.592217
6   cluster_id_t50       0.50        4424          3.802893    0.578793
7   cluster_id_t55       0.55        4022          4.182994    0.560654
8   cluster_id_t60       0.60        3638          4.624519    0.537681
9   cluster_id_t65       0.65        3272          5.141809    0.511417
10  cluster_id_t70       0.70        2891          5.819440    0.480768
11  cluster_id_t75       0.75        2499          6.732293    0.440680
12  cluster_id_t80       0.80        2042          8.238981    0

In [3]:
# import matplotlib.pyplot as plt

# plt.figure()

# plt.plot(df_scan["t"], df_scan["mean_cluster_size_sgo"], label="mean_size SGO")
# plt.plot(df_scan["t"], df_scan["mean_cluster_size_map4"], label="mean_size MAP4")

# plt.plot(df_scan["t"], df_scan["max_cluster_size_sgo"], label="max_size SGO")
# plt.plot(df_scan["t"], df_scan["max_cluster_size_map4"], label="max_size MAP4")

# plt.xlabel("t")
# plt.ylabel("cluster statistics")
# plt.yscale("log")
# plt.legend()

# plt.show()

## Compare MAP4 and SGO

In [7]:
# Keep fragmented mols with at least 2 annotated fragments (methyls and acetyls are neglected)
df_fragments = pd.read_pickle("/home/mattia/geass/NPA_fragmentation/NPA_fragments_map4.pkl")
df_frag_stats = df_fragments[df_fragments['Frag_ISOSMILES'].apply(lambda x: len(x) > 1)]
df_frag_stats = df_frag_stats[['NPA_ID', 'ISOSMILES', 'Frag_IDs', 'Frag_ISOSMILES']]

# df_frag_stats = df_frag_stats[['NPA_ID', 'ISOSMILES', 'Frag_IDs', 'Frag_ISOSMILES']]
# df_frag_stats['N_MU'] = df_frag_stats['Frag_IDs'].apply(
#     lambda lst: sum(x.startswith(('C', 'L')) for x in lst))
# df_frag_stats['N_U'] = df_frag_stats['Frag_IDs'].apply(
#     lambda lst: sum(x.startswith('U') for x in lst))

# alk = {"C00019", "C00033", "C00132"}
# df_frag_stats["N_alk"] = df_frag_stats["Frag_IDs"].apply(
#     lambda lst: sum(x in alk for x in lst))
# df_frag_stats['N_non_alk'] = df_frag_stats['Frag_IDs'].apply(
#     lambda lst: sum(x not in alk for x in lst))

# mask = ~((df_frag_stats['N_non_alk'] == 1) & (df_frag_stats['N_alk'] >= 1))
# df_frag_stats = df_frag_stats[mask]
# mask = df_frag_stats['N_non_alk'] != df_frag_stats['N_U']
# df_frag_stats = df_frag_stats[mask]
# mask = df_frag_stats['N_U'] < df_frag_stats['N_MU']
# df_frag_stats = df_frag_stats[mask]

# PandasTools.AddMoleculeColumnToFrame(df_frag_stats, 'ISOSMILES')
# df_frag_stats['ISOSMILES'] = df_frag_stats['ROMol'].apply(
#     lambda mol: Chem.MolToSmiles(mol, isomericSmiles=False, canonical=True))
# df_frag_stats = df_frag_stats.drop_duplicates(subset='ISOSMILES')

In [10]:
df_clusters_map4 = pd.read_csv("/home/mattia/geass/NPA_analysis/NPA_clusters_map4.csv").fillna("0")
df_clusters_sgo = pd.read_csv("/home/mattia/geass/NPA_analysis/NPA_clusters_sgo.csv").fillna("0")

# df_clusters_map4 = df_clusters_map4[
#     df_clusters_map4['NPA_ID'].isin(df_frag_stats['NPA_ID'])].fillna(0)
df_clusters_sgo = df_clusters_sgo[
    df_clusters_sgo['NPA_ID'].isin(df_frag_stats['NPA_ID'])].fillna(0)

In [ ]:
# from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

# map4_cols = [c for c in df_clusters_map4.columns if c.startswith("cluster_id_t")]
# sgo_cols = [c for c in df_clusters_sgo.columns if c.startswith("cluster_id_t")]

# rows = []

# for cluster_col_sgo in sgo_cols:
#     largest_cluster = df_clusters_sgo[cluster_col_sgo].value_counts().idxmax()

#     df_sgo_struct = df_clusters_sgo[df_clusters_sgo[cluster_col_sgo] != largest_cluster][
#         ["NPA_ID", cluster_col_sgo]
#     ].rename(columns={cluster_col_sgo: "cluster_id_sgo"})

#     for cluster_col_map4 in map4_cols:
#         df_map4_tmp = df_clusters_map4[["NPA_ID", cluster_col_map4]].rename(
#             columns={cluster_col_map4: "cluster_id_map4"}
#         )

#         df_cmp = df_sgo_struct.merge(df_map4_tmp, on="NPA_ID", how="inner")

#         labels_sgo = df_cmp["cluster_id_sgo"].to_numpy()
#         labels_map4 = df_cmp["cluster_id_map4"].to_numpy()

#         same_sgo = labels_sgo[:, None] == labels_sgo
#         same_map4 = labels_map4[:, None] == labels_map4

#         np.fill_diagonal(same_sgo, False)
#         np.fill_diagonal(same_map4, False)

#         intersection = (same_sgo & same_map4).sum()
#         union = (same_sgo | same_map4).sum()
#         jaccard_pairs = intersection / union if union > 0 else np.nan

#         ari = adjusted_rand_score(labels_sgo, labels_map4)
#         ami = adjusted_mutual_info_score(labels_sgo, labels_map4)

#         rows.append({
#             "cluster_col_sgo": cluster_col_sgo,
#             "cluster_col_map4": cluster_col_map4,
#             "n_shared": len(df_cmp),
#             "jaccard_pairs": jaccard_pairs,
#             "ari": ari,
#             "ami": ami
#         })

# df_scan_overlap = pd.DataFrame(rows)

# heatmap_df = df_scan_overlap.pivot(
#     index="cluster_col_sgo", columns="cluster_col_map4", values="jaccard_pairs")

# import matplotlib.pyplot as plt
# fig, ax = plt.subplots()
# im = ax.imshow(heatmap_df.values, cmap="RdYlGn")
# ax.set_xticks(range(len(heatmap_df.columns)), heatmap_df.columns, rotation=90)
# ax.set_yticks(range(len(heatmap_df.index)), heatmap_df.index)
# ax.set_xlabel("MAP4 clustering threshold")
# ax.set_ylabel("SGO clustering threshold")
# fig.colorbar(im, ax=ax, label="Jaccard overlap of co-clustered pairs")
# plt.tight_layout()
# plt.show()

In [ ]:
cluster_col_map4 = "cluster_id_t70"
cluster_col_sgo = "cluster_id_t75"

largest_cluster = df_clusters_sgo[cluster_col_sgo].value_counts().idxmax()

df_sgo_struct = df_clusters_sgo[df_clusters_sgo[cluster_col_sgo] != largest_cluster].copy()

df_cmp = df_sgo_struct[["NPA_ID", cluster_col_sgo]].merge(
    df_clusters_map4[["NPA_ID", cluster_col_map4]],
    on="NPA_ID", how="inner")
df_cmp = df_cmp.rename(columns={f"{cluster_col_sgo}": "cluster_id_sgo",
    f"{cluster_col_map4}": "cluster_id_map4"})

In [ ]:
rows = []

for cid, group in df_cmp.groupby("cluster_id_map4"):
    counts = group["cluster_id_sgo"].value_counts()

    purity = counts.iloc[0] / counts.sum()

    rows.append({
        "cluster_id_map4": cid,
        "size": counts.sum(),
        "purity": purity,
        "n_sgo_clusters": len(counts)
    })

df_purity_map4 = pd.DataFrame(rows)

print("mean purity:", df_purity_map4["purity"].mean())
print("median purity:", df_purity_map4["purity"].median())
print("clusters perfectly matching:", (df_purity_map4["purity"] == 1).sum())

low_purity = df_purity_map4[df_purity_map4["purity"] < 0.8]
print("low-purity clusters:", len(low_purity))

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt

# # df_cmp must contain:
# # 'cluster_id_map4'
# # 'cluster_id_sgo'

# # MAP4 -> SGO
# map4_to_sgo = (
#     df_cmp.groupby('cluster_id_map4')['cluster_id_sgo']
#     .nunique()
# )

# # SGO -> MAP4
# sgo_to_map4 = (
#     df_cmp.groupby('cluster_id_sgo')['cluster_id_map4']
#     .nunique()
# )

# print("\nMAP4 → SGO")
# print(map4_to_sgo.value_counts().sort_index())

# print("\nSGO → MAP4")
# print(sgo_to_map4.value_counts().sort_index())

In [ ]:
df_cmp = df_cmp.sort_values("cluster_id_sgo").copy()
df_cmp["cluster_id_sgo_seq"] = pd.factorize(df_cmp["cluster_id_sgo"])[0] + 1

# find dominant SGO cluster for each MAP4 cluster
map4_to_sgo = (
    df_cmp.groupby("cluster_id_map4")["cluster_id_sgo_seq"]
    .agg(lambda x: x.value_counts().idxmax())
)

# attach dominant SGO to MAP4 clusters
map4_map = map4_to_sgo.rename("dominant_sgo").reset_index()

# within each SGO cluster, enumerate MAP4 clusters
map4_map["map4_rank"] = (
    map4_map.groupby("dominant_sgo")
    .cumcount()
)

# create new MAP4 id aligned with SGO blocks
map4_map["cluster_id_map4_seq"] = (
    map4_map["dominant_sgo"] * 100 + map4_map["map4_rank"]
)

# merge back
df_cmp = df_cmp.merge(
    map4_map[["cluster_id_map4", "cluster_id_map4_seq"]],
    on="cluster_id_map4",
    how="left"
)

In [ ]:
# import numpy as np

# plt.figure(figsize=(8,6))

# plt.scatter(
#     df_cmp["cluster_id_map4_seq"] + np.random.uniform(-0.2,0.2,len(df_cmp)),
#     df_cmp["cluster_id_sgo"] + np.random.uniform(-0.2,0.2,len(df_cmp)),
#     s=2,
#     alpha=0.2
# )

# plt.xlabel("MAP4 cluster ID")
# plt.ylabel("SGO cluster ID")
# plt.title("MAP4 vs SGO cluster assignment")

# plt.show()

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

cid = low_purity.sample(1)[["cluster_id_map4"]].values[0][0]
subset = df_cmp[df_cmp["cluster_id_map4"] == cid]
subset = subset.merge(df_frag_stats[["NPA_ID", "ISOSMILES"]], on="NPA_ID", how="left")
mols = [Chem.MolFromSmiles(s) for s in subset["ISOSMILES"]]
legends = [f"MAP4:{m} | SGO:{s}" for s, m in zip(subset["cluster_id_sgo"], subset["cluster_id_map4"])]
img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(250, 250), legends=legends)
img

In [ ]:
# df_fragments
# display(Chem.MolFromSmiles(subset.loc[0, 'ISOSMILES']), Chem.MolFromSmiles(subset.loc[1, 'ISOSMILES']))
subset

In [ ]:
sys.path.append('/home/mattia/geass')
import sgo_similarity as sgo
engine = sgo.SgoSimilarity()
from new_geass import Geass
geass = Geass(method='map4')

# smia = 'C[C@]1(CCC[C@]2([C@H]1CC=C3[C@@H]2C[C@]45C(O4)[C@@H]([C@]6(C([C@@]5(C3)O)O6)COCCCC)O)C)CO'
# smib = 'C[C@]1(CCC[C@]2([C@H]1CC=C3[C@@H]2C[C@]45C(O4)[C@@H]([C@]6(C([C@@]5(C3)O)O6)COC)O)C)CO'
smia = subset.loc[2, 'ISOSMILES']
smib = subset.loc[1, 'ISOSMILES']
mola = Chem.MolFromSmiles(smia)
molb = Chem.MolFromSmiles(smib)
fragsa = geass.get_fragments(mola)
ga = geass.create_connection_graph(img=False)
fragsb = geass.get_fragments(molb)
gb = geass.create_connection_graph(img=False)
ga_data = engine.preprocess_graph(ga)
gb_data = engine.preprocess_graph(gb)
print(engine.score_pair(ga_data, gb_data))

# print(engine.score_pair_by_id('NPA014342', 'NPA003985'))

In [ ]:
display(ga_data, gb_data)
display(mola, molb)
# display(*geass.get_fragments(mola))


In [ ]:
# subsett = df_cmp[df_cmp['cluster_id_sgo'] == 219]
subsett = df_cmp[df_cmp['cluster_id_map4'] == 416]
subsett = subsett.merge(df_frag_stats, on='NPA_ID')
mols = [Chem.MolFromSmiles(s) for s in subsett['ISOSMILES']]

# from rdkit.Chem import AllChem
# AllChem.Compute2DCoords(mols[0])
# for m in mols[1:]:
#   print(m)
#   _ = AllChem.GenerateDepictionMatching2DStructure(m,mols[0])

img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(250, 250))
img

## Visualise dendrogram

In [ ]:
# Create dendrogram network
G = nx.Graph()
n = Z.shape[0] + 1

# Add leaves with labels
npa_ids = df_clusters['NPA_ID'].tolist()
for i, row in enumerate(df_clusters.itertuples()):
    G.add_node(i, type='leaf', label=row.NPA_ID, smiles=row.ISOSMILES)

# Add internal nodes
for i in range(Z.shape[0]):
    node_id = n + i
    left = int(Z[i, 0])
    right = int(Z[i, 1])
    height = float(Z[i, 2])

    G.add_node(node_id, height=height)
    G.add_edge(node_id, left)
    G.add_edge(node_id, right)

coords_2d = nx.nx_agraph.graphviz_layout(G, prog='sfdp', args='-GK=1.8')

In [ ]:
cluster_col = 'cluster_id_t70'

df_clusters["color_code"] = df_clusters[cluster_col]
all_idx = np.arange(sim_matrix.shape[0])
cluster_members = df_clusters.groupby(cluster_col).indices

# Grey-out unfragmented compounds (largest cluster)
largest_cluster = df_clusters[cluster_col].value_counts().idxmax()
df_clusters.loc[df_clusters[cluster_col] == largest_cluster, "color_code"] = 0

# Grey-out isolated clusters (no external similarity)
isolated_clusters = []
for cluster_id, member_idx in cluster_members.items():
    other_idx = np.setdiff1d(all_idx, member_idx)
    max_external_sim = sim_matrix[np.ix_(member_idx, other_idx)].max()
    if max_external_sim == 0.0:
        isolated_clusters.append(cluster_id)

df_clusters.loc[df_clusters[cluster_col].isin(isolated_clusters), "color_code"] = 0

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from faerun import Faerun

# Build shared node and edge data
nodes = list(G.nodes())
leaf_nodes = {n for n in nodes if G.nodes[n].get("type") == "leaf"}

color_codes = df_clusters["color_code"].tolist()
c = [color_codes[n] if n in leaf_nodes else 0 for n in nodes]
n_colours = int(df_clusters["color_code"].max())
rainbow = plt.cm.rainbow(np.linspace(0, 1, max(n_colours, 1)))
colours = np.vstack(([0.65, 0.65, 0.65, 1.0], rainbow))
cmap = ListedColormap(colours)

# Randomise cluster colour
cluster_ids = df_clusters[cluster_col].to_numpy()
unique_clusters = np.unique(cluster_ids)
shuffled = np.random.permutation(unique_clusters)
cluster_to_color = {cid: i + 1 for i, cid in enumerate(shuffled)}
random_colors = [cluster_to_color[cid] for cid in cluster_ids]
c = [random_colors[n] if n in leaf_nodes else 0 for n in nodes]

node_index = {n: i for i, n in enumerate(nodes)}
edges = list(G.edges())
edge_start = [node_index[u] for u, v in edges]
edge_end = [node_index[v] for u, v in edges]

s = [2 if n in leaf_nodes else 0 for n in nodes]
labels = [G.nodes[n].get("smiles", "") if n in leaf_nodes else "" for n in nodes]
info = [G.nodes[n].get("label", "") if n in leaf_nodes else "" for n in nodes]

# Save 2D Faerun plot
x_2d = [coords_2d[n][0] for n in nodes]
y_2d = [coords_2d[n][1] for n in nodes]

f = Faerun(view="front", coords=False)
f.add_scatter("nodes", {"x": x_2d, "y": y_2d, "c": c, "s": s,
    "labels": labels, "info": info}, colormap=cmap)
f.add_tree("edges", {"from": edge_start, "to": edge_end},
    point_helper="nodes")
f.plot("NPA_map4_2d", template="smiles")

clear_output()

In [ ]:
fname = 'NPA_map4_2d'
# fname = 'NPA_dendrogram_3d'

with open(f"{fname}.html", encoding='utf-8') as fh:
    html = fh.read()

replacements = [
    (
        """          this.el.search.addEventListener('click', e => {
            let searchTerm = prompt('Please enter a search term:', '');
            const results = this.search(searchTerm);

            for (const [name, indices] of Object.entries(results)) {
              if (!name in this.ohIndexMap) return;
              indices.forEach(index => {
                this.octreeHelpers[this.ohIndexMap[name]].addSelected(index);
              });
            }
            
            e.preventDefault();
            return false;
          });""",
        """          this.el.search.addEventListener('click', e => {
            const searchTerm = prompt('Please enter an NPA ID:', '');
            this.searchAndFocus(searchTerm);
            
            e.preventDefault();
            return false;
          });"""
    ),
    (
        """          window.addEventListener('keydown', e => {
            if ((e.keyCode == 114) || (e.ctrlKey && e.keyCode == 70)) {
              this.search();
              e.preventDefault();
              return false;
            }
          });""",
        """          window.addEventListener('keydown', e => {
            if ((e.keyCode == 114) || (e.ctrlKey && e.keyCode == 70)) {
              const searchTerm = prompt('Please enter an NPA ID:', '');
              this.searchAndFocus(searchTerm);
              e.preventDefault();
              return false;
            }
          });"""
    ),
    (
        """        search(term) {
          let results = {}
          let re = new RegExp(term, 'i');

          Object.keys(data).forEach(name => {
            if (!('labels' in data[name])) return;
            results[name] = []
            data[name]['labels'].forEach((label, i) => {
              if (re.test(label))
                results[name].push(i)
            });
          });

          return results;
        }""",
        """        clearSelection() {
          let items = [...this.selectedItems];
          items.forEach(item => {
            this.octreeHelpers[item.source].removeSelected(item.item.index);
          });
        }

        searchAndFocus(term) {
          let results = this.search(term);
          let firstMatch = null;

          this.clearSelection();

          for (const [name, indices] of Object.entries(results)) {
            if (!(name in this.ohIndexMap)) continue;

            indices.forEach(index => {
              if (firstMatch === null) {
                firstMatch = { name, index };
              }

              this.octreeHelpers[this.ohIndexMap[name]].addSelected(index);
            });
          }

          if (firstMatch === null) {
            if ((term || '').trim()) {
              alert('No molecules matched that NPA ID.');
            }
            return;
          }

          let selectedIndex = this.getSelectedIndex(
            this.ohIndexMap[firstMatch.name],
            firstMatch.index
          );

          if (selectedIndex !== null) {
            this.updateSelected(selectedIndex);
          }
        }

        search(term) {
          let results = {};
          let normalizedTerm = (term || '').trim().toLowerCase();

          Object.keys(data).forEach(name => {
            if (!('info' in data[name])) return;
            results[name] = [];
            if (!normalizedTerm) return;

            data[name]['info'].forEach((label, i) => {
              if ((label || '').toLowerCase().includes(normalizedTerm)) {
                results[name].push(i);
              }
            });
          });

          return results;
        }"""
    ),
    (
        """        setSelectedContent(fullLabel, labelIndex, selectedLabels, img) {
          this.el.selectedContainer.innerHTML = '';
          if (img)
            this.el.selectedContainer.appendChild(img);

          fullLabel.forEach((l, i) => {
            if (i === labelIndex) return;
            if (selectedLabels && selectedLabels[i]) {
              this.el.selectedContainer.appendChild(
                Faerun.createElement('div', { classes: 'label', content: selectedLabels[i] })
              );
            }
            this.el.selectedContainer.appendChild(
              Faerun.createElement('div', { classes: 'content', content: l })
            );
          });

          // Update the indicator
          this.updateSelectedIndicators();
        }""",
        """        setSelectedContent(fullInfo, selectedLabels, img) {
          this.el.selectedContainer.innerHTML = '';
          if (img)
            this.el.selectedContainer.appendChild(img);

          fullInfo.forEach((l, i) => {
            if (selectedLabels && selectedLabels[i]) {
              this.el.selectedContainer.appendChild(
                Faerun.createElement('div', { classes: 'label', content: selectedLabels[i] })
              );
            }
            this.el.selectedContainer.appendChild(
              Faerun.createElement('div', { classes: 'content', content: l })
            );
          });

          // Update the indicator
          this.updateSelectedIndicators();
        }"""
    ),
    (
        """          let fullLabel = data[phName].labels[item.item.index].split('__');

          let labelIndex = meta.label_index[seriesState];
          let titleIndex = meta.title_index[seriesState];
          let selectedLabels = meta.selected_labels[seriesState];

          SmilesDrawer.parse(fullLabel[labelIndex], tree => {
            smilesDrawer.draw(tree, 'smiles-canvas', 'dark', false);
            let img = Faerun.createElement('img');
            img.src = this.el.smilesCanvas.toDataURL();
            
            this.el.selectedCurrent.innerHTML = current + 1;
            this.el.selectedTotal.innerHTML = n;
            this.el.selectedTitle.innerHTML = fullLabel[titleIndex];

            this.setSelectedContent(fullLabel, labelIndex, selectedLabels, img);
          }, err => { console.log(err); });""",
        """          let fullLabel = data[phName].labels[item.item.index].split('__');
          let fullInfo = data[phName].info[item.item.index].split('__');

          let labelIndex = meta.label_index[seriesState];
          let selectedLabels = meta.selected_labels[seriesState];

          SmilesDrawer.parse(fullLabel[labelIndex], tree => {
            smilesDrawer.draw(tree, 'smiles-canvas', 'dark', false);
            let img = Faerun.createElement('img');
            img.src = this.el.smilesCanvas.toDataURL();
            
            this.el.selectedCurrent.innerHTML = current + 1;
            this.el.selectedTotal.innerHTML = n;
            this.el.selectedTitle.innerHTML = fullInfo[0] || '';

            this.setSelectedContent(fullInfo, selectedLabels, img);
          }, err => { console.log(err); });"""
    ),
    (
        """      #selected #selected-container {
        padding: 10px;
        font-size: 0.8em;
      }

      #selected #selected-container img {
        margin-left: -10px;
      }""",
        """      #selected #selected-container {
        padding: 10px;
        font-size: 0.8em;
        user-select: text;
        -webkit-user-select: text;
      }

      #selected #selected-container img {
        margin-left: -10px;
      }"""
    ),
    (
        """      .selected-indicator {
        position: absolute;
        z-index: 999;
        pointer-events: none;
      }

      .selected-indicator .crosshair-x {
        position: absolute;
        top: 0; left: 0; bottom: 0; right: 0;
        height: 2px;
        width: 100%;
        margin: auto;
        
          background-color: #fff;
        
      }

      .selected-indicator .crosshair-y {
        position: absolute;
        top: 0; left: 0; bottom: 0; right: 0;
        width: 2px;
        height: 100%;
        margin: auto;
        
          background-color: #fff;
        
      }""",
        """      .selected-indicator {
        position: absolute;
        z-index: 999;
        pointer-events: none;
      }

      .selected-indicator .selected-ring {
        position: absolute;
        top: -6px;
        left: -6px;
        width: 200%;
        height: 200%;
        border: 2px solid #fff;
        border-radius: 50%;
        box-sizing: border-box;
      }

      .selected-indicator .crosshair-x {
        position: absolute;
        top: 0; left: 0; bottom: 0; right: 0;
        height: 2px;
        width: 100%;
        margin: auto;
        
          background-color: #fff;
        
      }

      .selected-indicator .crosshair-y {
        position: absolute;
        top: 0; left: 0; bottom: 0; right: 0;
        width: 2px;
        height: 100%;
        margin: auto;
        
          background-color: #fff;
        
      }"""
    ),
    (
        """          let indicatorElement = Faerun.createElement(
            'div', 
            { classes: 'selected-indicator' },
            [
              Faerun.createElement('div', { classes: 'crosshair-x' }),
              Faerun.createElement('div', { classes: 'crosshair-y' })
            ]
          );""",
        """          let indicatorElement = Faerun.createElement(
            'div', 
            { classes: 'selected-indicator' },
            [
              Faerun.createElement('div', { classes: 'selected-ring' }),
              Faerun.createElement('div', { classes: 'crosshair-x' }),
              Faerun.createElement('div', { classes: 'crosshair-y' })
            ]
          );"""
    )
]

for old, new in replacements:
    if old not in html:
        raise ValueError('Expected HTML snippet not found while patching html file')
    html = html.replace(old, new, 1)

with open(f"{fname}.html", 'w', encoding='utf-8') as fh:
    fh.write(html)